In [ ]:
import numpy as np
import pickle as pkl
import matplotlib.pyplot as plt
import lsst.summit.utils.butlerUtils as butlerUtils
from lsst.summit.utils.efdUtils import makeEfdClient, getEfdData, getMostRecentRowWithDataBefore
from lsst.summit.utils.butlerUtils import getExpRecordFromDataId

In [ ]:
client = makeEfdClient()
butler = butlerUtils.makeDefaultButler("LSSTCam", embargo=True)

In [ ]:
day_obs = 20260404
firstSeqNum = 14
lastSeqNum = 646
firstExpId = int(day_obs*1e5 + firstSeqNum)
lastExpId = int(day_obs*1e5 + lastSeqNum)
dataId = {'exposure': firstExpId, 'instrument':'LSSTCam'}
firstExpRecord = getExpRecordFromDataId(butler, dataId)
dataId = {'exposure': lastExpId, 'instrument':'LSSTCam'}
lastExpRecord = getExpRecordFromDataId(butler, dataId)

In [ ]:
wavefront_errors =getEfdData(client, 
    topic="lsst.sal.MTAOS.logevent_wavefrontError",
    columns=[
        f"nollZernikeValues{i}" for i in range(27)
    ] + [
        f"nollZernikeIndices{i}" for i in range(27)
    ] + [
        "sensorId",
        "visitId"
    ],
    begin=firstExpRecord.timespan.begin,
    end=lastExpRecord.timespan.end                         
)
print(len(wavefront_errors))

dof_event = getEfdData(
    client,
    topic="lsst.sal.MTAOS.logevent_degreeOfFreedom",
    columns=['visitDoF0', 'visitDoF5', 'visitId'],
    begin=firstExpRecord.timespan.begin,
    end=lastExpRecord.timespan.end                         
)
print(len(dof_event))

In [ ]:
exposureList = []
for record in butler.registry.queryDimensionRecords("exposure", 
                    where=f"exposure.day_obs={day_obs} and instrument='LSSTCam'"):
    exposureList.append([record.id, record])
exposureList.sort(key=lambda x: x[0])
z4s = []
m2dzs = []
camdzs = []
for [expId, record] in exposureList:
    seqNum = int(expId - 1e5*day_obs)
    try:
        these_wavefront_errors = wavefront_errors[wavefront_errors['visitId']==expId]
        this_dof_event = dof_event[dof_event['visitId']==expId]
        #print(seqNum, len(these_wavefront_errors), len(this_dof_event))
        z4 = these_wavefront_errors['nollZernikeValues0'].values.mean()
        m2dz = this_dof_event['visitDoF0'].values[0]
        camdz = this_dof_event['visitDoF5'].values[0]
        
        
        z4s.append(z4)
        m2dzs.append(m2dz)
        camdzs.append(camdz)
    except:
        continue
print(len(z4s), len(m2dzs), len(camdzs))
        


In [ ]:
fig, axes = plt.subplots(1,3,figsize=(15,5))
fig.suptitle(f"Z4 correlations {day_obs}")
axes[0].scatter(m2dzs, z4s)
axes[0].set_title("M2 dz")
axes[0].set_ylabel("Z4 (microns)")
axes[0].set_xlabel("M2 dz (microns)")
axes[0].set_ylim(-1.5,1)
axes[0].set_xlim(-75,75)
axes[1].scatter(camdzs, z4s)
axes[1].set_title("Cam dz")
axes[1].set_ylabel("Z4 (microns)")
axes[1].set_xlabel("Cam dz (microns)")
axes[1].set_ylim(-1.5,1)
axes[1].set_xlim(-75,75)
vm = 0.6 * np.array(m2dzs) + 0.4 * np.array(camdzs)
axes[2].scatter(vm, z4s)
axes[2].set_title("Vmode")
axes[2].set_ylabel("Z4 (microns)")
axes[2].set_xlabel("Vmode (microns)")
axes[2].set_ylim(-1.5,1)
axes[2].set_xlim(-75,75)
fig.savefig(f"/home/c/cslage/u/MTAOS/times_square_reports/Z4_Correlations_{day_obs}.png")